# 2. Byte-Pair Encoding (BPR) Tokenizer

## 2.1 The Unicode Standard

In [ ]:
1772177181.2118177 - 1772177179.3979626

In [1]:
# What Unicode character does chr(0) return?
# Answer: Null Character
chr(0) 


'\x00'

In [2]:
#  How does this character’s string representation (__repr__()) differ from its printed representation? 
repr('\x00')
# print 没有办法显示 "\x00" 
# repr能够显示

"'\\x00'"

In [3]:
# What happens when this character occurs in text? It may be helpful to play around with the
# following in your Python interpreter and see if it matches your expectations:
chr(0) 
print(chr(0))

 


In [4]:
"this is a test" + chr(0) + "string" 
print("this is a test" + chr(0) + "string")
# 当这个字符出现在文本中时，它是一个不可见的空字符，虽然实际存在于字符串中，但打印时不会显示任何可见内容。

this is a test string


In [1]:
# unicode character
test_string = "hello! こんにちは!" 
# Get the byte values for the encoded string (integers from 0 to 255)
utf8_encoded = test_string.encode("utf - 8") 
print(utf8_encoded)  # 16进制 hexadecimal
print(type(utf8_encoded)) 

print(list(utf8_encoded)) # 10进制 decimal

b'hello! \xe3\x81\x93\xe3\x82\x93\xe3\x81\xab\xe3\x81\xa1\xe3\x81\xaf!'
<class 'bytes'>
[104, 101, 108, 108, 111, 33, 32, 227, 129, 147, 227, 130, 147, 227, 129, 171, 227, 129, 161, 227, 129, 175, 33]


In [6]:
list(test_string.encode("utf-8"))

[104,
 101,
 108,
 108,
 111,
 33,
 32,
 227,
 129,
 147,
 227,
 130,
 147,
 227,
 129,
 171,
 227,
 129,
 161,
 227,
 129,
 175,
 33]

In [4]:
a = list(map(int,test_string.encode("utf-8")) )
print(a)

[104, 101, 108, 108, 111, 33, 32, 227, 129, 147, 227, 130, 147, 227, 129, 171, 227, 129, 161, 227, 129, 175, 33]


In [6]:
print(chr(33))
print(ord("!"))

!
33


In [7]:
# # One byte does not necessarily correspond to one Unicode character!
print(test_string)
print(len(utf8_encoded))

hello! こんにちは!
23


## 2.2 Unicode Encoding

In [8]:
# What are some reasons to prefer training our tokenizer on UTF-8 encoded bytes, rather than
# UTF-16 or UTF-32? It may be helpful to compare the output of these encodings for various
# input strings. 

test_string = "hello! こんにちは!"  
utf16_encoded = test_string.encode("utf - 16") 
print(utf16_encoded) 
print(list(utf16_encoded)) 
print(len(utf16_encoded)) 

# utf 16/32 为定长编码, 每个字符4 bytes 
# utf 8 为变长编码, code point 0-127 为1 byte, 128-2047 为2bytes, 其他为3-4 bytes

b'\xff\xfeh\x00e\x00l\x00l\x00o\x00!\x00 \x00S0\x930k0a0o0!\x00'
[255, 254, 104, 0, 101, 0, 108, 0, 108, 0, 111, 0, 33, 0, 32, 0, 83, 48, 147, 48, 107, 48, 97, 48, 111, 48, 33, 0]
28


In [9]:
test_string = "hello! こんにちは!"  
utf32_encoded = test_string.encode("utf - 32") 
print(utf32_encoded) 
print(list(utf32_encoded)) 
print(len(utf32_encoded))

b'\xff\xfe\x00\x00h\x00\x00\x00e\x00\x00\x00l\x00\x00\x00l\x00\x00\x00o\x00\x00\x00!\x00\x00\x00 \x00\x00\x00S0\x00\x00\x930\x00\x00k0\x00\x00a0\x00\x00o0\x00\x00!\x00\x00\x00'
[255, 254, 0, 0, 104, 0, 0, 0, 101, 0, 0, 0, 108, 0, 0, 0, 108, 0, 0, 0, 111, 0, 0, 0, 33, 0, 0, 0, 32, 0, 0, 0, 83, 48, 0, 0, 147, 48, 0, 0, 107, 48, 0, 0, 97, 48, 0, 0, 111, 48, 0, 0, 33, 0, 0, 0]
56


In [47]:
initial_vocab = list()
for i in range(256): 
    initial_vocab.append(bytes([i])) 

print(initial_vocab)

[b'\x00', b'\x01', b'\x02', b'\x03', b'\x04', b'\x05', b'\x06', b'\x07', b'\x08', b'\t', b'\n', b'\x0b', b'\x0c', b'\r', b'\x0e', b'\x0f', b'\x10', b'\x11', b'\x12', b'\x13', b'\x14', b'\x15', b'\x16', b'\x17', b'\x18', b'\x19', b'\x1a', b'\x1b', b'\x1c', b'\x1d', b'\x1e', b'\x1f', b' ', b'!', b'"', b'#', b'$', b'%', b'&', b"'", b'(', b')', b'*', b'+', b',', b'-', b'.', b'/', b'0', b'1', b'2', b'3', b'4', b'5', b'6', b'7', b'8', b'9', b':', b';', b'<', b'=', b'>', b'?', b'@', b'A', b'B', b'C', b'D', b'E', b'F', b'G', b'H', b'I', b'J', b'K', b'L', b'M', b'N', b'O', b'P', b'Q', b'R', b'S', b'T', b'U', b'V', b'W', b'X', b'Y', b'Z', b'[', b'\\', b']', b'^', b'_', b'`', b'a', b'b', b'c', b'd', b'e', b'f', b'g', b'h', b'i', b'j', b'k', b'l', b'm', b'n', b'o', b'p', b'q', b'r', b's', b't', b'u', b'v', b'w', b'x', b'y', b'z', b'{', b'|', b'}', b'~', b'\x7f', b'\x80', b'\x81', b'\x82', b'\x83', b'\x84', b'\x85', b'\x86', b'\x87', b'\x88', b'\x89', b'\x8a', b'\x8b', b'\x8c', b'\x8d', b'\x8e', b'

In [10]:
# Consider the following (incorrect) function, which is intended to 
# decode a UTF-8 byte string into a Unicode string. 
# Why is this function incorrect? 
# Provide an example of an input byte string that yields incorrect results. 

def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])

In [11]:
decode_utf8_bytes_to_str_wrong("hello".encode("utf-8")) 
# hello encode之后为bytestring
# 原函数是一个byte 一个byte 进行decode
# hello 中每一个字符占据一个byte, 所以原函数可以进行decode
# 但是不是所有语言, 每个字符只有一个byte, 例如中文 
# 所以无法一个byte 一个byte 进行decode, 必须整体进行decode

'hello'

# 2.3 Subword Tokenization

Subword tokenization is a midpoint between word-level tokenizers and byte-level tokenizers. Note that a byte-level tokenizer’s vocabulary has 256 entries (byte values are 0 to 225). A subword tokenizer trades-off a larger vocabulary size for better compression of the input byte sequence. 

For example, if the byte sequence 'the' often occurs in our raw text training data, assigning it an entry in the vocabulary would reduce this 3-token sequence to a single token

# 2.4 BPE TOkenizer Training

## Pre tokenization 

You can think of this as a coarse-grained tokenization over the corpus that helps us count how often pairs of characters appear. For example, the word 'text' might be a pre-token that appears 10 times. 

In this case, when we count how often the characters ‘t’ and ‘e’ appear
next to each other, we will see that the word ‘text’ has ‘t’ and ‘e’ adjacent and we can increment their count by 10 instead of looking through the corpus. 

In [1]:
import regex as re
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

In [2]:
re.findall(PAT, "some text that i'll pre-tokenize")

['some', ' text', ' that', ' i', "'ll", ' pre', '-', 'tokenize']

In [3]:
token = list() 
corpus = {}
for m in re.finditer(PAT, "some text that i'll pre-tokenize"):
    token = m.group().encode("utf-8")
    token_byte = tuple(bytes([b]) for b in token)
    corpus[token_byte]= corpus.get(token_byte, 0) + 1  
 
corpus

{(b's', b'o', b'm', b'e'): 1,
 (b' ', b't', b'e', b'x', b't'): 1,
 (b' ', b't', b'h', b'a', b't'): 1,
 (b' ', b'i'): 1,
 (b"'", b'l', b'l'): 1,
 (b' ', b'p', b'r', b'e'): 1,
 (b'-',): 1,
 (b't', b'o', b'k', b'e', b'n', b'i', b'z', b'e'): 1}

In [6]:
pairs = {} 
for word, freq in corpus.items(): 
    for i in range(len(word) - 1): 
        pair = (word[i], word[i + 1]) 
        pairs [pair] = pairs.get(pair, 0) + freq  

pairs

{(b's', b'o'): 1,
 (b'o', b'm'): 1,
 (b'm', b'e'): 1,
 (b' ', b't'): 2,
 (b't', b'e'): 1,
 (b'e', b'x'): 1,
 (b'x', b't'): 1,
 (b't', b'h'): 1,
 (b'h', b'a'): 1,
 (b'a', b't'): 1,
 (b' ', b'i'): 1,
 (b"'", b'l'): 1,
 (b'l', b'l'): 1,
 (b' ', b'p'): 1,
 (b'p', b'r'): 1,
 (b'r', b'e'): 1,
 (b't', b'o'): 1,
 (b'o', b'k'): 1,
 (b'k', b'e'): 1,
 (b'e', b'n'): 1,
 (b'n', b'i'): 1,
 (b'i', b'z'): 1,
 (b'z', b'e'): 1}

In [12]:
items = list(pairs.items())
items

[((b's', b'o'), 1),
 ((b'o', b'm'), 1),
 ((b'm', b'e'), 1),
 ((b' ', b't'), 2),
 ((b't', b'e'), 1),
 ((b'e', b'x'), 1),
 ((b'x', b't'), 1),
 ((b't', b'h'), 1),
 ((b'h', b'a'), 1),
 ((b'a', b't'), 1),
 ((b' ', b'i'), 1),
 ((b"'", b'l'), 1),
 ((b'l', b'l'), 1),
 ((b' ', b'p'), 1),
 ((b'p', b'r'), 1),
 ((b'r', b'e'), 1),
 ((b't', b'o'), 1),
 ((b'o', b'k'), 1),
 ((b'k', b'e'), 1),
 ((b'e', b'n'), 1),
 ((b'n', b'i'), 1),
 ((b'i', b'z'), 1),
 ((b'z', b'e'), 1)]

In [15]:
token = list()
for m in re.finditer(PAT, "some text that i'll pre-tokenize"): 
    print("word:", m.group()) 
    print("location:", m.span())
    print(m.group()) 
    print(list(m.group())) 
    token.append(list(m.group()))
 
print(token)

word: some
location: (0, 4)
some
['s', 'o', 'm', 'e']
word:  text
location: (4, 9)
 text
[' ', 't', 'e', 'x', 't']
word:  that
location: (9, 14)
 that
[' ', 't', 'h', 'a', 't']
word:  i
location: (14, 16)
 i
[' ', 'i']
word: 'll
location: (16, 19)
'll
["'", 'l', 'l']
word:  pre
location: (19, 23)
 pre
[' ', 'p', 'r', 'e']
word: -
location: (23, 24)
-
['-']
word: tokenize
location: (24, 32)
tokenize
['t', 'o', 'k', 'e', 'n', 'i', 'z', 'e']
[['s', 'o', 'm', 'e'], [' ', 't', 'e', 'x', 't'], [' ', 't', 'h', 'a', 't'], [' ', 'i'], ["'", 'l', 'l'], [' ', 'p', 'r', 'e'], ['-'], ['t', 'o', 'k', 'e', 'n', 'i', 'z', 'e']]


In [27]:
vocab = {}
for m in re.finditer(PAT, "some text that i'll pre-tokenize"): 
        token = m.group()  
        char = list(token) 
        vocab[tuple(char)]= vocab.get(tuple(char), 0) + 1  
print(vocab)

{('s', 'o', 'm', 'e'): 1, (' ', 't', 'e', 'x', 't'): 1, (' ', 't', 'h', 'a', 't'): 1, (' ', 'i'): 1, ("'", 'l', 'l'): 1, (' ', 'p', 'r', 'e'): 1, ('-',): 1, ('t', 'o', 'k', 'e', 'n', 'i', 'z', 'e'): 1}


In [43]:
pairs= dict()
for key, count in vocab.items(): 
        for i in range(len(key) - 1): 
            pair = (key[i], key[i + 1])
            pairs[pair] = pairs.get(pair, 0) + 1  

keys_lists = list(pairs.keys())
print(keys_lists)


[('s', 'o'), ('o', 'm'), ('m', 'e'), (' ', 't'), ('t', 'e'), ('e', 'x'), ('x', 't'), ('t', 'h'), ('h', 'a'), ('a', 't'), (' ', 'i'), ("'", 'l'), ('l', 'l'), (' ', 'p'), ('p', 'r'), ('r', 'e'), ('t', 'o'), ('o', 'k'), ('k', 'e'), ('e', 'n'), ('n', 'i'), ('i', 'z'), ('z', 'e')]


In [16]:
print(type(re.finditer(PAT, "some text that i'll pre-tokenize")))

<class '_regex.Scanner'>


## Compute BPE merges

In [1]:
# When computing merges, deterministically break ties in pair frequency by
# preferring the lexicographically greater pair
# A < B < C < ... < Z
max([("A", "B"), ("A", "C"), ("B", "ZZ"), ("BA", "A")])

('BA', 'A')

## Special Tokens 

In [3]:
# <|endoftext|>  文本结束
# <pad> batch 对齐长度
# <bos>  开始
# <eos>  结束 
# When encoding text, it’s often desirable to treat some strings as “special tokens” that
# should never be split into multiple tokens (i.e., will always be preserved as a single token. 

In [6]:
# \d 数字0-9 
# \D 非数字
# \w 字母+ 数字+ 下划线
# \W 非字母数字
# \s 空格
# \S 非空格

In [ ]:
corpus = "low low low low low lower lower widest widest widest newest newest newest newest newest newest" 

In [8]:
import regex as re 
for m in re.finditer(r"\S+", corpus): 
    print("word:", m.group()) 
    print("location:", m.span())

word: low
location: (0, 3)
word: low
location: (4, 7)
word: low
location: (8, 11)
word: low
location: (12, 15)
word: low
location: (16, 19)
word: lower
location: (20, 25)
word: lower
location: (26, 31)
word: widest
location: (32, 38)
word: widest
location: (39, 45)
word: widest
location: (46, 52)
word: newest
location: (53, 59)
word: newest
location: (60, 66)
word: newest
location: (67, 73)
word: newest
location: (74, 80)
word: newest
location: (81, 87)
word: newest
location: (88, 94)


## 2.5 Experimenting with BPE Tokenizer Training

In [ ]:
import time, pickle, sys, os

# 1. 设置路径 (tests 就在当前目录)
sys.path.insert(0, os.path.abspath("tests"))
from adapters import run_train_bpe 

# 2. 配置参数 (针对 TinyStories Valid)
data_dir = "data"
# 输入路径
tinystories_valid_path = os.path.join(data_dir, "TinyStoriesV2-GPT4-valid.txt")
vocab_size = 10000
special_tokens = ["<|endoftext|>"]

# 3. 运行并计时
print(f"正在开始 BPE 训练 (TinyStories Valid): {tinystories_valid_path} ...")
t0 = time.time()
vocab, merges = run_train_bpe(tinystories_valid_path, vocab_size, special_tokens)
dt = (time.time() - t0) / 60

# 4. 统计与展示
longest = max(vocab.values(), key=len)
print("-" * 30)
print(f"TinyStories Valid 训练完成！耗时: {dt:.2f} 分钟")
print(f"最长 Token: {longest.decode('utf-8', errors='replace')} ({len(longest)} 字节)")

# 5. 保存到 data 文件夹 (文件名加入 _valid 后缀以示区分)
vocab_file = os.path.join(data_dir, "tinystories_vocab_valid.pkl")
merges_file = os.path.join(data_dir, "tinystories_merges_valid.pkl")

with open(vocab_file, "wb") as f: 
    pickle.dump(vocab, f)
with open(merges_file, "wb") as f: 
    pickle.dump(merges, f)

print(f"结果已保存至:\n - {vocab_file}\n - {merges_file}")

正在开始 BPE 训练 (TinyStories Valid): data\TinyStoriesV2-GPT4-valid.txt ...
------------------------------
TinyStories Valid 训练完成！耗时: 0.82 分钟
最长 Token:  accomplishment (15 字节)
结果已保存至:
 - data\tinystories_vocab_valid.pkl
 - data\tinystories_merges_valid.pkl


In [7]:
import time, pickle, sys, os

# 1. 环境配置
sys.path.insert(0, os.path.abspath("tests"))
from adapters import run_train_bpe 

# 2. 配置参数 (针对 TinyStories Train)
data_dir = "data"

tinystories_train_path = os.path.join(data_dir, "TinyStoriesV2-GPT4-train.txt")
vocab_size = 10000
special_tokens = ["<|endoftext|>"]

# 3. 运行并计时
print(f"正在开始 BPE 训练 (TinyStories Train): {tinystories_train_path}")

t0 = time.time()


vocab, merges = run_train_bpe(tinystories_train_path, vocab_size, special_tokens)

dt = (time.time() - t0) / 60

# 4. 统计与展示
longest = max(vocab.values(), key=len)
print("-" * 30)
print(f"总耗时: {dt:.2f} 分钟")
print(f"最长 Token: {longest.decode('utf-8', errors='replace')} ({len(longest)} 字节)")

# 5. 保存到 data 文件夹 (文件名改为 _train)
vocab_file = os.path.join(data_dir, "tinystories_vocab_train.pkl")
merges_file = os.path.join(data_dir, "tinystories_merges_train.pkl")

with open(vocab_file, "wb") as f: 
    pickle.dump(vocab, f)
with open(merges_file, "wb") as f: 
    pickle.dump(merges, f)

print(f"结果已保存至:\n - {vocab_file}\n - {merges_file}")

正在开始 BPE 训练 (TinyStories Train): data\TinyStoriesV2-GPT4-train.txt
------------------------------
总耗时: 26.15 分钟
最长 Token:  accomplishment (15 字节)
结果已保存至:
 - data\tinystories_vocab_train.pkl
 - data\tinystories_merges_train.pkl
